### TRANSFERIR LOS DATOS A LA TABLA DE HECHOS DE LA CAPA GOLD
**IMPORTAMOS LAS LIBRERIAS**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**EXTRAEMOS LOS DATOS**

In [0]:
gold_silver = spark.table("proyecto_spotify.silver.spotify_tracks")
#display(gold_silver)

In [0]:
dim_artist = spark.table("proyecto_spotify.gold.dim_artista")
dim_album = spark.table("proyecto_spotify.gold.dim_album")
dim_track = spark.table("proyecto_spotify.gold.dim_track")
dim_date = spark.table("proyecto_spotify.gold.dim_fecha")

**UNIMOS LOS DATOS DE LA CAPA SILVER CON LOS DATOS DE LAS TABLAS DIMENSIONES**

In [0]:
fact_track = (
    gold_silver.alias("s")

    .join(
        dim_track.alias("t"),
        col("s.track_id") == col("t.track_id"),
        "left"
    )

    .join(
        dim_artist.alias("a"),
        col("s.artist_id") == col("a.artist_id"),
        "left"
    )

    .join(
        dim_album.alias("al"),
        col("s.album_id") == col("al.album_id"),
        "left"
    )

    .join(
        dim_date.alias("d"),
        col("s.release_date_clean") == col("d.full_date"),
        "left"
    )
)

#display(fact_track.limit(10))

**SELECCIONAMOS LOS DATOS NECESARIOS PARA LA TABLA DE HECHOS**

In [0]:
fact_track = fact_track.select(
    
    # Claves dimensionales
    col("t.sk_track"),
    col("a.sk_artist"),
    col("al.sk_album"),
    col("d.sk_date"),

    # Claves naturales
    col("s.track_id"),
    col("s.artist_id"),
    col("s.album_id"),

    # Métricas
    col("s.popularity"),
    col("s.popularity_level"),
    col("s.duration_minutes"),

    # Indicadores
    col("s.explicit"),

    # Información temporal
    col("s.release_date"),
    col("s.release_date_precision"),   
    col("s.release_date_clean"),

    # Información de extracción
    col("s.extraction_timestamp"),
    col("s.search_term")
)

#display(fact_track)

**GUARDAMOS LOS DATOS EN LA TABLA FACT_TRACK DE LA CAPA GOLD**

In [0]:
(
    fact_track
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "proyecto_spotify.gold.fact_track"
    )
)